# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")

# 1. Distribution of target variable
print(fraud['fraud'].value_counts())
print(fraud['fraud'].value_counts(normalize=True))

In [3]:
# 2. Train Logistic Regression (baseline, imbalanced)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X = fraud.drop(columns=['fraud'])
y = fraud['fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0)

log_reg = LogisticRegression(max_iter=1000, random_state=0)
log_reg.fit(X_train, y_train)

# 3. Evaluate — accuracy is misleading here, use classification_report instead
pred = log_reg.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

[[181343   1272]
 [  6899  10486]]
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182615
         1.0       0.89      0.60      0.72     17385

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000



In [4]:
# 4. Oversampling
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=0)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

log_reg_over = LogisticRegression(max_iter=1000, random_state=0)
log_reg_over.fit(X_train_over, y_train_over)
pred_over = log_reg_over.predict(X_test)
print("=== Oversampled ===")
print(classification_report(y_test, pred_over))

=== Oversampled ===
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182615
         1.0       0.57      0.95      0.71     17385

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [5]:
# 5. Undersampling
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=0)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

log_reg_under = LogisticRegression(max_iter=1000, random_state=0)
log_reg_under.fit(X_train_under, y_train_under)
pred_under = log_reg_under.predict(X_test)
print("=== Undersampled ===")
print(classification_report(y_test, pred_under))

=== Undersampled ===
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182615
         1.0       0.57      0.95      0.71     17385

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [6]:
# 6. SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=0)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

log_reg_smote = LogisticRegression(max_iter=1000, random_state=0)
log_reg_smote.fit(X_train_smote, y_train_smote)
pred_smote = log_reg_smote.predict(X_test)
print("=== SMOTE ===")
print(classification_report(y_test, pred_smote))

=== SMOTE ===
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182615
         1.0       0.58      0.95      0.72     17385

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [ ]:
# The baseline model trained on imbalanced data achieved high overall accuracy (96%) 
#but poor fraud recall (60%) — missing 40% of actual fraud cases, which is the more costly 
#error in this context. Both undersampling and SMOTE dramatically improved fraud recall to 95%, 
#at the cost of lower precision (more false positives) and slightly lower overall accuracy (93%). 
#Given that failing to catch fraud is more costly than flagging a legitimate transaction for review,
# this tradeoff is worthwhile — undersampling and SMOTE both meaningfully outperform the baseline on
#the metric that actually matters for this business problem. Interestingly, undersampling performed 
#nearly identically to SMOTE despite being the simpler technique, discarding majority-class data 
#rather than generating synthetic minority examples.